In [1]:
import os

# Move working directory to project root
os.chdir("E:/HealthyBites")

print("Now working directory is:", os.getcwd())


Now working directory is: E:\HealthyBites


In [3]:
import pandas as pd
import ast
from collections import Counter

print("Loading dataset...")
df = pd.read_csv("data/processed/healthybites_master_dataset.csv")

# Convert ingredients column to list safely
def safe_eval(x):
    try:
        if isinstance(x, list):
            return [str(i).lower().strip() for i in x]
        return [str(i).lower().strip() for i in ast.literal_eval(x)]
    except:
        return []

df["ingredients"] = df["ingredients"].apply(safe_eval)

print("Total recipes:", len(df))

Loading dataset...
Total recipes: 200000


In [4]:
print("Calculating ingredient frequency...")

all_ingredients = []
for ing_list in df["ingredients"]:
    all_ingredients.extend(set(ing_list))  # use set to avoid double counting per recipe

ingredient_counts = Counter(all_ingredients)

total_recipes = len(df)

# Convert to frequency percentage
ingredient_freq = {
    ing: count / total_recipes
    for ing, count in ingredient_counts.items()
}

Calculating ingredient frequency...


In [8]:
# Fixed pantry list (manually defined staples)
FIXED_PANTRY = {
    "salt", "pepper", "black pepper", "water", "garlic", "tomato",
    "oil", "olive oil", "vegetable oil", "butter", "onion",
    "sugar", "flour", "baking soda", "baking powder",
    "vinegar", "soy sauce", "garlic powder",
    "onion powder", "paprika", "cumin",
    "turmeric", "oregano", "thyme",
    "chili powder"
}

# Frequency threshold (you can adjust)
FREQUENCY_THRESHOLD = 0.35   # appears in >35% recipes → pantry

In [9]:
print("Classifying core vs pantry...")

def split_ingredients(ingredient_list):
    core = []
    pantry = []

    for ing in ingredient_list:
        if ing in FIXED_PANTRY:
            pantry.append(ing)
        elif ingredient_freq.get(ing, 0) > FREQUENCY_THRESHOLD:
            pantry.append(ing)
        else:
            core.append(ing)

    return core, pantry

df["core_ingredients"] = df["ingredients"].apply(lambda x: split_ingredients(x)[0])
df["pantry_ingredients"] = df["ingredients"].apply(lambda x: split_ingredients(x)[1])

Classifying core vs pantry...


In [10]:
df.to_csv("data/processed/healthybites_master_dataset_split.csv", index=False)

print("New split dataset saved successfully!")

New split dataset saved successfully!
